# PCFG Word Set Generator
Generates an exhaustive password set from a trained PCFG ruleset.

Reads from `ruleset.json`, applies selected structural patterns with lowercase-only cap mask, 
and writes all unique combinations to a single output file.

---

## Cell 1 -- Configuration
Set the ruleset path and select which structural patterns to generate from.

To change patterns later, edit `SELECTED_PATTERNS` and re-run from Cell 2.

In [1]:
import os
import json

# ── CONFIGURE THIS ────────────────────────────────────────────────
RULESET_PATH = "pcfg_output/pcfg_len8/ruleset.json"  # change to your ruleset path

# Patterns to generate from -- add or remove as needed
# Format must match structural tags in your ruleset exactly
SELECTED_PATTERNS = [
    "L8",
    "L6D2",
    "L7D1",
    "L5D3",
]
# ─────────────────────────────────────────────────────────────────

# Auto-derive output path from ruleset path
# pcfg_output/pcfg_len8/ruleset.json -> wordset_output/wordset_len8.txt
length_tag  = os.path.basename(os.path.dirname(RULESET_PATH))  # pcfg_len8
len_suffix  = length_tag.replace("pcfg_", "")                  # len8
OUTPUT_DIR  = "wordset_output"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"wordset_{len_suffix}.txt")

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.isfile(RULESET_PATH):
    raise FileNotFoundError(f"Ruleset not found: {RULESET_PATH}\nCheck your RULESET_PATH above.")

print(f"Input ruleset  : {RULESET_PATH}")
print(f"Output file    : {OUTPUT_FILE}")
print(f"Patterns       : {SELECTED_PATTERNS}")

Input ruleset  : pcfg_output/pcfg_len8/ruleset.json
Output file    : wordset_output/wordset_len8.txt
Patterns       : ['L8', 'L6D2', 'L7D1', 'L5D3']


## Cell 2 -- Load Ruleset
Loads the trained ruleset and extracts the components needed for generation.

In [2]:
with open(RULESET_PATH, 'r', encoding='utf-8') as f:
    ruleset = json.load(f)

# Extract base words indexed by length
# Keys may be ints or strings depending on JSON serialisation -- normalise to int
base_words_by_length = {
    int(k): [r['value'] for r in v]
    for k, v in ruleset.get('base_words_by_length', {}).items()
}

# Extract digit strings indexed by length
digit_strings_by_length = {}
for r in ruleset.get('digit_strings', []):
    l = len(r['value'])
    digit_strings_by_length.setdefault(l, []).append(r['value'])

# Validate selected patterns exist in ruleset
ruleset_structures = {r['value'] for r in ruleset.get('structures', [])}
print(f"Structures in ruleset  : {len(ruleset_structures)}")
print()
for pat in SELECTED_PATTERNS:
    status = 'OK' if pat in ruleset_structures else 'WARNING -- NOT FOUND IN RULESET'
    print(f"  [{status}]  {pat}")

print(f"\nBase word lengths available   : {sorted(base_words_by_length.keys())}")
print(f"Digit string lengths available: {sorted(digit_strings_by_length.keys())}")

Structures in ruleset  : 300

  [OK]  L8
  [OK]  L6D2
  [OK]  L7D1
  [OK]  L5D3

Base word lengths available   : [1, 2, 3, 4, 5, 6, 7, 8]
Digit string lengths available: [1, 2, 3, 4, 5, 6]


## Cell 3 -- Size Estimate
Calculates how many passwords will be generated per pattern **before** generating anything.

Review this before proceeding to Cell 4.

In [3]:
import re

def parse_pattern(pattern):
    """
    Parse structural tag into segments.
    'L6D2' -> [('L', 6), ('D', 2)]
    'L8'   -> [('L', 8)]
    """
    return [(t, int(n)) for t, n in re.findall(r'([LDS])(\d+)', pattern)]

print(f"  {'Pattern':<10} {'Base Words':>12} {'Digit Strings':>15} {'Combinations':>15}")
print(f"  {'─'*10}  {'─'*12}  {'─'*15}  {'─'*15}")

grand_total   = 0
pattern_sizes = {}

for pat in SELECTED_PATTERNS:
    segs   = parse_pattern(pat)
    l_segs = [(t, n) for t, n in segs if t == 'L']
    d_segs = [(t, n) for t, n in segs if t == 'D']

    bw_count = 1
    for _, n in l_segs:
        bw_count *= len(base_words_by_length.get(n, []))

    ds_count = 1
    for _, n in d_segs:
        ds_count *= len(digit_strings_by_length.get(n, []))

    total = bw_count * ds_count
    pattern_sizes[pat] = total
    grand_total += total

    ds_display = str(ds_count) if d_segs else 'N/A'
    print(f"  {pat:<10} {bw_count:>12,} {ds_display:>15} {total:>15,}")

print(f"  {'─'*10}  {'─'*12}  {'─'*15}  {'─'*15}")
print(f"  {'TOTAL':<10} {'':>12} {'':>15} {grand_total:>15,}")
print(f"\n  Estimated file size : {grand_total * 10 / 1_000_000:.1f} MB")
print(f"\n  NOTE: Output file will be overwritten if it already exists.")

  Pattern      Base Words   Digit Strings    Combinations
  ──────────  ────────────  ───────────────  ───────────────
  L8              718,744             N/A         718,744
  L6D2            202,476             100      20,247,600
  L7D1            169,506              10       1,695,060
  L5D3             70,697             221      15,624,037
  ──────────  ────────────  ───────────────  ───────────────
  TOTAL                                        38,285,441

  Estimated file size : 382.9 MB

  NOTE: Output file will be overwritten if it already exists.


## Cell 4 -- Generate Password Set
Generates all combinations for each selected pattern and writes them to the output file.

Uses streaming via `itertools.product` so memory usage stays low regardless of output size.

Output file is always overwritten on re-run.

In [4]:
import itertools

def generate_pattern(pattern, base_words_by_length, digit_strings_by_length):
    """
    Exhaustively generate all lowercase combinations for a given structural pattern.
    Yields one password string at a time to keep memory usage low.
    """
    segs = parse_pattern(pattern)
    slot_options = []

    for seg_type, seg_len in segs:
        if seg_type == 'L':
            words = base_words_by_length.get(seg_len, [])
            if not words:
                print(f"  WARNING: No base words of length {seg_len} -- skipping pattern {pattern}")
                return
            # Lowercase only cap mask applied here
            slot_options.append([w.lower() for w in words])
        elif seg_type == 'D':
            digits = digit_strings_by_length.get(seg_len, [])
            if not digits:
                print(f"  WARNING: No digit strings of length {seg_len} -- skipping pattern {pattern}")
                return
            slot_options.append(digits)

    for combo in itertools.product(*slot_options):
        yield ''.join(combo)


print(f"Writing to: {OUTPUT_FILE}\n")

total_written = 0

with open(OUTPUT_FILE, 'w', encoding='utf-8') as fout:
    for pat in SELECTED_PATTERNS:
        pat_count = 0
        expected  = pattern_sizes.get(pat, '?')
        print(f"  Generating {pat:<10} (expected {expected:,}) ...", end='', flush=True)

        for password in generate_pattern(pat, base_words_by_length, digit_strings_by_length):
            fout.write(password + '\n')
            pat_count     += 1
            total_written += 1

        print(f" done -- {pat_count:,} written")

print(f"\nTotal passwords written : {total_written:,}")
print(f"Output file size        : {os.path.getsize(OUTPUT_FILE) / 1_000_000:.1f} MB")
print(f"Saved to                : {OUTPUT_FILE}")

Writing to: wordset_output/wordset_len8.txt

  Generating L8         (expected 718,744) ... done -- 718,744 written
  Generating L6D2       (expected 20,247,600) ... done -- 20,247,600 written
  Generating L7D1       (expected 1,695,060) ... done -- 1,695,060 written
  Generating L5D3       (expected 15,624,037) ... done -- 15,624,037 written

Total passwords written : 38,285,441
Output file size        : 344.7 MB
Saved to                : wordset_output/wordset_len8.txt


## Cell 5 -- Verify Output
Spot-checks the output file -- confirms line count, length distribution, 
and shows the first and last few passwords as a sanity check.

In [5]:
from collections import Counter

line_count  = 0
length_dist = Counter()
first_lines = []

with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        pw = line.rstrip('\n')
        line_count += 1
        length_dist[len(pw)] += 1
        if line_count <= 5:
            first_lines.append(pw)

with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    last_lines = [l.rstrip('\n') for l in f.readlines()[-5:]]

print(f"Total lines    : {line_count:,}")
print(f"File size      : {os.path.getsize(OUTPUT_FILE) / 1_000_000:.1f} MB")
print(f"\nLength distribution:")
for l, c in sorted(length_dist.items()):
    print(f"  {l} chars : {c:,}")
print(f"\nFirst 5 passwords:")
for pw in first_lines:
    print(f"  {pw}")
print(f"\nLast 5 passwords:")
for pw in last_lines:
    print(f"  {pw}")

Total lines    : 38,285,441
File size      : 344.7 MB

Length distribution:
  8 chars : 38,285,441

First 5 passwords:
  password
  iloveyou
  princess
  jennifer
  michelle

Last 5 passwords:
  ttwwl138
  ttwwl156
  ttwwl621
  ttwwl821
  ttwwl326


In [6]:
import json

with open("pcfg_output/pcfg_len8/ruleset.json", "r") as f:
    ruleset = json.load(f)

# Count base words per length
print("Base words per length:")
for length, words in sorted(ruleset['base_words_by_length'].items(), key=lambda x: int(x[0])):
    print(f"  Length {length} : {len(words):,} words")

# Count digit strings per length
from collections import Counter
digit_len_counts = Counter(len(r['value']) for r in ruleset['digit_strings'])
print("\nDigit strings per length:")
for l, c in sorted(digit_len_counts.items()):
    print(f"  Length {l} : {c:,} strings")

Base words per length:
  Length 1 : 95 words
  Length 2 : 725 words
  Length 3 : 14,835 words
  Length 4 : 68,363 words
  Length 5 : 70,697 words
  Length 6 : 202,476 words
  Length 7 : 169,506 words
  Length 8 : 718,744 words

Digit strings per length:
  Length 1 : 10 strings
  Length 2 : 100 strings
  Length 3 : 221 strings
  Length 4 : 167 strings
  Length 5 : 1 strings
  Length 6 : 1 strings


In [1]:
print("Base words per length:")
for length, words in sorted(ruleset['base_words_by_length'].items(), key=lambda x: int(x[0])):
    print(f"  Length {length} : {len(words):,} words")
    

Base words per length:


NameError: name 'ruleset' is not defined